# Learning Objectives

* Build a per-radar-file Pegasus workflow from prebuilt radar tools in an Apptainer container.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the gridded hydrometeor classification produced by a run.

# CASA Hail Workflow

This workflow runs CASA (Collaborative Adaptive Sensing of the Atmosphere) hail detection on weather radar
scans. Each radar moment file runs an independent pipeline that classifies hydrometeors (hail/rain/snow) per
sweep and maps the classification onto a cartesian grid; independent files run in parallel. The radar name
prefix selects the hydroclass configuration and grid parameters. See `README.md` in this repository for the
full description and the legacy DAX3 history.

**Workflow Jobs (per radar file):**
1. `gunzip` - decompress the file if it is `.gz`
2. `RadxConvert` - NEXRAD (`burleson.tx`, KFWS) only: rewrite Level-II data to CfRadial
3. `hydroclass` - classify hydrometeors per sweep, using the radar's config and the current sounding
4. `Radx2Grid` - map the classification onto a cartesian grid
5. `netcdf2png` - CASA X-band radars only: render the classification to a PNG

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Sample Input Data

A sample NEXRAD scan ships in `input/`: `burleson.tx-20170329-071410.netcdf` (KFWS, 2017-03-29 07:14 UTC,
from the public NOAA archive — the DFW hail event also used by the casa-wind test data), alongside the
hydroclass configs (`HC_*.ini`), per-radar grid parameters (`*_latlon.txt`), the atmospheric sounding, and the
color scale.

In [ ]:
!ls -l input/

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/CASA_Hail_Container.def`) converted from the published
`casaelyons/single_hail_container` Docker image, which ships the tools prebuilt. Pegasus stages the `.sif` to the workers, so jobs need
no registry access. Building it can take a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/CASA_Hail_Container.sif ]; then
    apptainer build Apptainer/CASA_Hail_Container.sif Apptainer/CASA_Hail_Container.def
else
    echo "Container already built: Apptainer/CASA_Hail_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `HailWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs, embedded in
`workflow.yml`) and writes it with `pegasus.properties`. Nothing about the workflow is defined here: change the
pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` adds a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import HailWorkflow, resolve_radar_file

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

RADAR_FILES = [resolve_radar_file(f) for f in ["burleson.tx-20170329-071410.netcdf"]]
print(f"Input files: {[Path(f).name for f in RADAR_FILES]}")

workflow = HailWorkflow(outdir=str(Path(".").resolve()), radar_files=RADAR_FILES)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating CASA Hail workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nCASA Hail Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. For the NEXRAD sample the graph is a single chain: `RadxConvert` → `hydroclass` → `Radx2Grid`; each additional radar file adds its own parallel chain.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (with `HOSTED_SITE_CATALOG` set, Pegasus's
default `wf-output/` instead):

| Output | Description |
|--------|-------------|
| `{radar}-{YYYYMMDD}-{HHMMSS}.hc.cart.netcdf` | Hydrometeor classification on a cartesian grid |
| `{radar}-{YYYYMMDD}-{HHMMSS}-hmc.png` | Rendered classification (CASA X-band radars only, not the NEXRAD sample) |

In [ ]:
!ls -lt output/

The gridded classification for the sample scan:

In [ ]:
!ls -l output/burleson.tx-20170329-071410.hc.cart.netcdf